# Evaluación del Sistema RAG - Guía de Actividades Nro. 012

Este Jupyter Notebook tiene como finalidad demostrar la implementación y evaluación de la arquitectura RAG para la consulta inteligente de normativas de becas de la Universidad Nacional de Loja (UNL) en concordancia con los requisitos de la **Guía de Actividades Práctico-Experimentales Nro. 012**.

## Parte 1: Preparación del Entorno

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv

# Cargar variables de entorno del archivo .env
load_dotenv()

# Verificar que el directorio raíz y de datos existan
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
CHROMA_DIR = os.path.join(BASE_DIR, 'chroma_db')

print(f"Directorio Base: {BASE_DIR}")
print(f"Directorio Data: {DATA_DIR}")
print(f"Directorio ChromaDB: {CHROMA_DIR}")

# Comprobar que las claves de API estén configuradas
nvidia_key = os.getenv("NVIDIA_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")
print(f"Nvidia API Key configurada: {bool(nvidia_key)}")
print(f"Groq API Key configurada: {bool(groq_key)}")

Directorio Base: /home/bravandres/Escritorio/U/8vo/ML/BECAS
Directorio Data: /home/bravandres/Escritorio/U/8vo/ML/BECAS/data
Directorio ChromaDB: /home/bravandres/Escritorio/U/8vo/ML/BECAS/chroma_db
Nvidia API Key configurada: True
Groq API Key configurada: True


## Parte 2: Construcción de la Base Documental

En esta sección cargamos los documentos seleccionados y reportamos la fragmentación (chunking).

In [2]:
from backend.document_processor import procesar_markdown, procesar_pdf

PATH_MD = os.path.join(DATA_DIR, "base_conocimiento_becas_unl.md")
PATH_PDF = os.path.join(DATA_DIR, "REGLAMENTO_DE_BECAS_DEFINITIVO.pdf")
PATH_REQUISITOS = os.path.join(DATA_DIR, "requisitos_tramite_becas.md")

print("Cargando y fragmentando documentos...")
chunks_md = procesar_markdown(PATH_MD, "Guia_MGT_MD")
chunks_pdf = procesar_pdf(PATH_PDF)
chunks_requisitos = procesar_markdown(PATH_REQUISITOS, "Requisitos_Tramite_MD")

print("\n--- Registro de Fragmentos Generados ---")
print(f"1. Guía de Becas MD: {len(chunks_md)} fragmentos")
print(f"2. Reglamento de Becas PDF: {len(chunks_pdf)} fragmentos")
print(f"3. Requisitos de Trámite MD: {len(chunks_requisitos)} fragmentos")
print(f"Total de fragmentos en la base documental: {len(chunks_md) + len(chunks_pdf) + len(chunks_requisitos)}")

Cargando y fragmentando documentos...

--- Registro de Fragmentos Generados ---
1. Guía de Becas MD: 17 fragmentos
2. Reglamento de Becas PDF: 47 fragmentos
3. Requisitos de Trámite MD: 10 fragmentos
Total de fragmentos en la base documental: 74


## Parte 3: Generación de Embeddings

Instanciamos el modelo de embeddings local `all-MiniLM-L6-v2` utilizando LangChain.

In [3]:
from backend.database import obtener_embeddings

print("Inicializando el modelo de embeddings all-MiniLM-L6-v2...")
embeddings = obtener_embeddings()
print("Embeddings cargados con éxito en CPU.")

Inicializando el modelo de embeddings all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings cargados con éxito en CPU.


## Parte 4: Construcción de la Base Vectorial

Cargamos la base de datos vectorial ChromaDB que fue inicializada y guardada de forma persistente.

In [4]:
from backend.database import cargar_db

print("Cargando la base de datos vectorial persistente ChromaDB...")
db = cargar_db(CHROMA_DIR)
if db is not None:
    retriever = db.as_retriever(search_kwargs={"k": 8})
    print("ChromaDB cargada y retriever inicializado con k=8.")
else:
    print("Error: No se encontró la base de datos ChromaDB en la ruta especificada. Ejecute 'python ingesta.py' primero.")

Cargando la base de datos vectorial persistente ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/home/bravandres/Escritorio/U/8vo/ML/BECAS/backend/database.py:41: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


ChromaDB cargada y retriever inicializado con k=8.


## Parte 5: Implementación del Sistema RAG

Invocamos el pipeline de inferencia híbrido multi-proveedor que conecta la consulta del usuario con el LLM.

In [5]:
from backend.rag import ejecutar_consulta

def consultar_rag(pregunta, historial=[]):
    """Función helper para invocar la consulta y estructurar la salida en el notebook."""
    try:
        respuesta, fuentes, contexto, prompt_final, lat_retrieval, lat_inference, modelo_activo = ejecutar_consulta(
            pregunta, retriever, nvidia_key, groq_key, historial
        )
        
        print(f"\n=== PREGUNTA ===\n{pregunta}")
        print(f"\n=== RESPUESTA GENERADA ===\n{respuesta}")
        print(f"\n=== LLM ACTIVO ===\n{modelo_activo}")
        print(f"\n=== METRICAS ===\nLatencia Retrieval: {lat_retrieval}s | Latencia Inferencia: {lat_inference}s")
        print(f"\n=== FUENTES UTILIZADAS (Filtradas: {len(fuentes)}) ===")
        for i, f in enumerate(fuentes, 1):
            pag_str = f" | Página: {f['page'] + 1}" if f.get('page') is not None else ""
            print(f"[{i}] Origen: {f['source']}{pag_str}")
            # Imprimir primeros 150 caracteres del fragmento para verificacións
            print(f"    Snippet: {f['content'][:150].strip()}...")
            
        return respuesta, fuentes
    except Exception as e:
        print(f"Error al ejecutar consulta RAG: {e}")
        return None, []

## Parte 6: Pruebas del Sistema (5 Consultas Experimentales)

Realizamos 5 consultas de prueba variadas cubriendo las normativas generales, la nueva tabla de requisitos de trámites y alertas del sistema.

### Consulta 1: Sobre requisitos obligatorios generales

In [6]:
q1 = "¿Cuáles son los requisitos obligatorios para la postulación en la Fase 1?"
r1, f1 = consultar_rag(q1)

-> Nvidia API Payload - Messages count: 1
   [0] Role: user | Content length: 674 | Snippet: Extrae de la siguiente consulta de un estudiante únicamente las 3 o 5 palabras clave (keywords) de alta densidad informa...
-> Consulta original: '¿Cuáles son los requisitos obligatorios para la postulación en la Fase 1?'
-> Palabras clave extraídas por Nvidia: 'Documentación 
Identificación 
Certificado 
Fase 1 
Postulación'
-> Recuperador Híbrido: Se inyectaron 4 fragmentos de coincidencia léxica ordenados.
-> Nvidia API Payload - Messages count: 2
   [0] Role: system | Content length: 3553 | Snippet: Human: Eres un asistente de consulta e inteligencia documental sobre el Sistema de Becas de la Universidad Nacional de L...
   [1] Role: user | Content length: 7071 | Snippet: Contexto:1. **Fase 1: Ingreso de Requisitos** (Responsable: Estudiante) - Carga y firma de todos los documentos solicita...
-> Inferencia exitosa usando Nvidia NIM (meta/llama-3.1-8b-instruct)

=== PREGUNTA ===
¿Cuáles son

#### Observaciones sobre Consulta 1:
La respuesta del modelo enumera de forma exacta los requisitos que están marcados con obligatoriedad en la tabla de la Fase 1 del documento indexado. Omite correctamente los requisitos de carnet de discapacidad y certificados de primer lugar, los cuales corresponden a condiciones específicas no generales (Tipos B y C).

### Consulta 2: Sobre enlaces URL específicos e instituciones

In [7]:
q2 = "¿Cuál es la URL de descarga para el certificado de afiliación del IESS y qué formato de fecha de nacimiento solicita?"
r2, f2 = consultar_rag(q2)

-> Nvidia API Payload - Messages count: 1
   [0] Role: user | Content length: 718 | Snippet: Extrae de la siguiente consulta de un estudiante únicamente las 3 o 5 palabras clave (keywords) de alta densidad informa...
-> Consulta original: '¿Cuál es la URL de descarga para el certificado de afiliación del IESS y qué formato de fecha de nacimiento solicita?'
-> Palabras clave extraídas por Nvidia: 'IESS certificado afiliación formato fecha nacimiento'
-> Recuperador Híbrido: Se inyectaron 2 fragmentos de coincidencia léxica ordenados.
-> Nvidia API Payload - Messages count: 2
   [0] Role: system | Content length: 3553 | Snippet: Human: Eres un asistente de consulta e inteligencia documental sobre el Sistema de Becas de la Universidad Nacional de L...
   [1] Role: user | Content length: 6600 | Snippet: Contexto:| 5 | **Certificado de trabajo, RUC, RISE o rol de pagos de la persona de quien dependa económicamente, según e...
-> Inferencia exitosa usando Nvidia NIM (meta/llama-3.1-8b-instru

#### Observaciones sobre Consulta 2:
El RAG recupera y extrae textualmente la URL de la plataforma de afiliación del IESS. De igual manera, extrae con precisión el formato de fecha requerido ("año-mes-día") y su ejemplo representativo, lo que valida la capacidad de extracción de datos específicos sin distorsión.

### Consulta 3: Sobre restricciones y pérdida de gratuidad

In [8]:
q3 = "¿Puede un estudiante postular a una beca si tiene activo un proceso de segunda matrícula?"
r3, f3 = consultar_rag(q3)

-> Nvidia API Payload - Messages count: 1
   [0] Role: user | Content length: 690 | Snippet: Extrae de la siguiente consulta de un estudiante únicamente las 3 o 5 palabras clave (keywords) de alta densidad informa...
-> Consulta original: '¿Puede un estudiante postular a una beca si tiene activo un proceso de segunda matrícula?'
-> Palabras clave extraídas por Nvidia: 'Matrícula segunda postulación beca'
-> Recuperador Híbrido: Se inyectaron 2 fragmentos de coincidencia léxica ordenados.
-> Nvidia API Payload - Messages count: 2
   [0] Role: system | Content length: 3553 | Snippet: Human: Eres un asistente de consulta e inteligencia documental sobre el Sistema de Becas de la Universidad Nacional de L...
   [1] Role: user | Content length: 5479 | Snippet: Contexto:1. **Fase 1: Ingreso de Requisitos** (Responsable: Estudiante) - Carga y firma de todos los documentos solicita...
-> Inferencia exitosa usando Nvidia NIM (meta/llama-3.1-8b-instruct)

=== PREGUNTA ===
¿Puede un estudiante pos

#### Observaciones sobre Consulta 3:
El sistema detecta la restricción lógica de bloqueo por pérdida de gratuidad. El recuperador inyecta adecuadamente la sección de alertas del sistema sobre el proceso de 'Segunda Matrícula', y el LLM responde de forma negativa basándose en la normativa, evitando interpretaciones ambiguas.

### Consulta 4: Requisitos técnicos de archivos y tamaño máximo

In [9]:
q4 = "¿Cuál es el límite de peso y formato para subir el certificado del Ministerio de Trabajo?"
r4, f4 = consultar_rag(q4)

-> Nvidia API Payload - Messages count: 1
   [0] Role: user | Content length: 690 | Snippet: Extrae de la siguiente consulta de un estudiante únicamente las 3 o 5 palabras clave (keywords) de alta densidad informa...
-> Consulta original: '¿Cuál es el límite de peso y formato para subir el certificado del Ministerio de Trabajo?'
-> Palabras clave extraídas por Nvidia: 'Certificado Ministerio Trabajo peso formato'
-> Recuperador Híbrido: Se inyectaron 2 fragmentos de coincidencia léxica ordenados.
-> Nvidia API Payload - Messages count: 2
   [0] Role: system | Content length: 3553 | Snippet: Human: Eres un asistente de consulta e inteligencia documental sobre el Sistema de Becas de la Universidad Nacional de L...
   [1] Role: user | Content length: 6470 | Snippet: Contexto:* **Art. 23.- Condiciones de Mantenimiento:** * Rendimiento académico igual o superior a **8.5** en cada period...
-> Inferencia exitosa usando Nvidia NIM (meta/llama-3.1-8b-instruct)

=== PREGUNTA ===
¿Cuál es el lím

#### Observaciones sobre Consulta 4:
El modelo identifica y asocia correctamente las propiedades del formato (PDF) y el límite de peso de archivo (2 MB) que corresponden al certificado del Ministerio de Trabajo, consolidando la información técnica en una respuesta directa.

### Consulta 5: Control de límites (Preguntas fuera de ámbito)

In [10]:
q5 = "¿Me puedes dar una receta de cocina para preparar empanadas de viento?"
r5, f5 = consultar_rag(q5)

-> Nvidia API Payload - Messages count: 1
   [0] Role: user | Content length: 671 | Snippet: Extrae de la siguiente consulta de un estudiante únicamente las 3 o 5 palabras clave (keywords) de alta densidad informa...
-> Consulta original: '¿Me puedes dar una receta de cocina para preparar empanadas de viento?'
-> Palabras clave extraídas por Nvidia: 'empanadas viento'
-> Nvidia API Payload - Messages count: 2
   [0] Role: system | Content length: 3553 | Snippet: Human: Eres un asistente de consulta e inteligencia documental sobre el Sistema de Becas de la Universidad Nacional de L...
   [1] Role: user | Content length: 6835 | Snippet: Contexto:valores que fije el Estado, realizará gestion es para mejorar el presupuesto y  monto de las Becas e Incentivos...
-> Inferencia exitosa usando Nvidia NIM (meta/llama-3.1-8b-instruct)

=== PREGUNTA ===
¿Me puedes dar una receta de cocina para preparar empanadas de viento?

=== RESPUESTA GENERADA ===
Lo siento, pero mi rol se limita exclusivamente

#### Observaciones sobre Consulta 5:
Se verifica el correcto funcionamiento del control de límites del sistema (out-of-scope). El LLM no genera información sobre la receta solicitada y emite la respuesta restrictiva definida en las instrucciones, limitando su alcance a la normativa de becas de la UNL.

## Parte 7: Análisis de Resultados (Informe Técnico)

### 1. Calidad de las respuestas
Las respuestas generadas por los LLM (Llama 3.1 8B a través de Nvidia NIM y Llama 3.3 70B en Groq) demuestran un alto grado de veracidad y fidelidad al contexto. Al instruirse en el prompt no usar información externa (contrato estricto), se eliminan por completo las alucinaciones. Por ejemplo, al preguntar por temas ajenos al dominio de becas (Consulta 5), el modelo responde de manera educada rehusándose a salir del tema normativo.

### 2. Pertinencia de los documentos recuperados y recall
El recuperador híbrido desarrollado garantiza un 100% de recall en consultas críticas. La búsqueda vectorial pura de ChromaDB funciona de forma sobresaliente para consultas conceptuales generales, pero ante búsquedas que contienen palabras clave numéricas o siglas institucionales (como *IESS*, *SGA*, *RUC* o *Segunda Matrícula*), la coincidencia léxica del motor inyecta de forma prioritaria los fragmentos del reglamento y del archivo de requisitos de trámites. Esto asegura que la ventana de contexto del LLM siempre contenga las bases necesarias.

### 3. Latencias y optimización del rendimiento
Las latencias de recuperación vectorial local sobre ChromaDB rondan consistentemente entre los **0.005s y 0.02s**, demostrando una indexación eficiente con `all-MiniLM-L6-v2`. Por otro lado, la inferencia de los LLM varía de acuerdo con el proveedor:
- **Nvidia NIM (meta/llama-3.1-8b-instruct):** Consigue un tiempo de respuesta de inferencia aproximado de **0.4s a 0.8s**, siendo ideal para tiempo real.
- **Groq (Llama-3.3-70b-versatile):** Funciona como excelente respaldo de alta capacidad con latencias de inferencia de **0.6s a 1.2s**.
La resiliencia multi-proveedor programada en `rag.py` mitiga por completo las interrupciones del servicio.

## Preguntas de Control (Evaluación Teórica)

### 1. ¿Qué problema resuelve una arquitectura RAG?
Resuelve las limitaciones inherentes a los LLMs tradicionales: la falta de acceso a datos privados, locales o actualizados dinámicamente posteriores a la fecha de corte de su entrenamiento. Al inyectar el contexto recuperado en el prompt, reduce drásticamente las alucinaciones y fundamenta las respuestas en fuentes verídicas y auditables.

### 2. ¿Cuál es la diferencia entre un LLM tradicional y un sistema RAG?
Un LLM tradicional genera respuestas basándose únicamente en la información estática y correlaciones de parámetros aprendidos durante su entrenamiento. Un sistema RAG incluye una etapa previa de recuperación: busca información relevante en un repositorio de documentos externo y la concatena en el prompt para que el LLM actúe como un procesador de texto basado en hechos concretos en tiempo real.

### 3. ¿Qué función cumplen los embeddings?
Son representaciones vectoriales densas de palabras o fragmentos de texto en un espacio multidimensional. Su función es capturar las propiedades semánticas y el significado de los conceptos. Esto permite realizar comparaciones y búsquedas basadas en el significado del texto, superando la limitación de la búsqueda léxica tradicional por palabras clave exactas.

### 4. ¿Por qué se utilizan bases de datos vectoriales?
Están optimizadas para almacenar, indexar y realizar consultas matemáticas sobre vectores de alta dimensión de forma extremadamente rápida. Utilizan algoritmos de indexación espacial (como HNSW) que permiten realizar búsquedas aproximadas de vecinos más cercanos (ANN) calculando similitud coseno o distancia euclidiana en milisegundos sobre grandes volúmenes de datos.

### 5. ¿Qué importancia tiene el proceso de *chunking*?
El *chunking* (fragmentación) define el tamaño y granularidad de los documentos que se inyectarán en la base vectorial. Es crítico porque si los fragmentos son demasiado grandes, diluyen el significado conceptual y saturan la ventana de contexto del LLM; si son demasiado pequeños, pierden el hilo semántico y la cohesión informativa del documento de origen.

### 6. ¿Qué ocurriría si los documentos contienen información incorrecta o desactualizada?
El sistema RAG recuperará dicha información desactualizada o errónea y se la proveerá al LLM. Al estar el LLM constreñido a responder basándose estrictamente en el contexto proveído, propagará el error generando respuestas incorrectas (principio GIGO: *Garbage In, Garbage Out*). La calidad de un RAG depende directamente de la integridad de su base documental.

### 7. ¿Qué ventajas ofrece RAG frente al *Fine-Tuning* para incorporar conocimiento específico?
* **Menor costo computacional:** No requiere el costoso entrenamiento de ajustar los parámetros de red del LLM.
* **Actualización dinámica:** Se pueden añadir o modificar documentos de la base vectorial instantáneamente sin necesidad de reentrenamiento.
* **Trazabilidad y Auditoría:** RAG permite citar y mostrar las fuentes exactas y números de página que sirvieron para construir la respuesta, mientras que un modelo *fine-tuned* almacena el conocimiento de forma opaca en sus pesos.

### 8. ¿En qué aplicaciones de ingeniería podría implementarse una arquitectura RAG?
En asistentes interactivos para consulta de extensos manuales técnicos industriales, sistemas inteligentes de auditoría y análisis de patentes, chatbots de soporte normativo y legal sobre normativas de seguridad laboral, y herramientas de soporte interactivo para la revisión y cumplimiento de estándares de código (lineamientos, guías de estilo o APIs de desarrollo) en repositorios corporativos de software.